<a href="https://colab.research.google.com/github/viswadevadi-spec/fraud-policy-assistant/blob/main/Fraud_policy_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain chromadb sentence-transformers gradio


In [ ]:
!pip install transformers torch accelerate bitsandbytes

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install -q langchain langchain-community chromadb langchain-text-splitters pypdf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/fraud_rag/docs", exist_ok=True)
os.makedirs("/content/drive/MyDrive/fraud_rag/chroma_db", exist_ok=True)

print("Drive mounted and folders ready")

In [ ]:
import urllib.request
import os

save_dir = "/content/drive/MyDrive/fraud_rag/docs"
os.makedirs(save_dir, exist_ok=True)

# working SEC.gov PDFs (April 2026)
pdfs = {
    "sec_2024_financial_report.pdf":
        "https://www.sec.gov/files/sec-2024-agency-financial-report.pdf",

    "sec_2026_exam_priorities.pdf":
        "https://www.sec.gov/files/2026-exam-priorities.pdf",

    "sec_whistleblower_2025.pdf":
        "https://www.sec.gov/files/fy25-annual-whistleblower-report.pdf",

    "sec_regulation_sp_compliance.pdf":
        "https://www.sec.gov/files/rules/final/2024/regulation-s-p-small-entity-compliance-guide.pdf",
}

headers = {'User-Agent': 'Mozilla/5.0 fraud-rag-project research@example.com'}

for filename, url in pdfs.items():
    path = f"{save_dir}/{filename}"
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        print(f"  Already exists: {filename}")
        continue
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=30) as response:
            with open(path, 'wb') as f:
                f.write(response.read())
        size_kb = os.path.getsize(path) / 1024
        print(f" Downloaded: {filename} ({size_kb:.0f} KB)")
    except Exception as e:
        print(f" Failed: {filename} → {e}")

print("\n Files in docs folder:")
for f in os.listdir(save_dir):
    size = os.path.getsize(f"{save_dir}/{f}") / 1024
    print(f"   {f} ({size:.0f} KB)")

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the documents from the folder
loader = PyPDFDirectoryLoader("/content/drive/MyDrive/fraud_rag/docs/")
docs = loader.load()
print(f" Loaded {len(docs)} pages")

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)
print(f" Created {len(chunks)} chunks")

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Embed
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.page_content for c in chunks]
embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

# Store — persisted to Drive, survives session restarts
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/fraud_rag/chroma_db"
)
collection = client.get_or_create_collection(
    name="fraud_policy",
    metadata={"hnsw:space": "cosine"}
)

# Only add if collection is empty
if collection.count() == 0:
    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        ids=[f"doc_{i}" for i in range(len(texts))]
    )
    print(f" Stored {len(texts)} chunks in ChromaDB")
else:
    print(f" ChromaDB already has {collection.count()} chunks — skipping")

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import bitsandbytes as bnb




In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Using Qwen2.5-3B which is open and fits well in Colab T4
model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f" Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# We name this 'model' to match your query_fraud_assistant function
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda",
    low_cpu_mem_usage=True
)
model.eval()

print(" Model loaded as 'model' — no login needed")

In [ ]:
def query_fraud_assistant(question: str) -> str:
    # Step 1: Embed the query
    q_embedding = embed_model.encode([question])[0].tolist()

    # Step 2: Retrieve top-5 chunks
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=5
    )
    context = "\n\n---\n\n".join(results['documents'][0])

    # Step 3: Build prompt
    prompt = f"""You are a fraud policy expert. Answer using ONLY the context below.
If the answer isn't in the context, say "I don't know." Don't hallucinate.

Context:
{context}

Question: {question}
Answer (with source references):"""

    # Step 4: Tokenize + Generate (Using the 'model' variable)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens
    answer = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return answer.strip()

# Quick test
if 'model' in globals():
    print(query_fraud_assistant("What is a Ponzi scheme?"))
else:
    print("Load the model in the cell above first!")

In [ ]:
import gradio as gr

def respond(question):
    if not question.strip():
        return "Please ask a question."
    return query_fraud_assistant(question)

demo = gr.Interface(
    fn=respond,
    inputs=gr.Textbox(
        label="Ask about fraud policy...",
        placeholder="e.g. What are red flags for securities fraud?",
        lines=3
    ),
    outputs=gr.Textbox(label="Answer", lines=10),
    title="🔍 Fraud Policy Assistant",
    description="RAG-powered compliance Q&A | Llama-3.2-3B + ChromaDB + SEC Docs",
    examples=[
        ["What is a Ponzi scheme?"],
        ["What are common signs of securities fraud?"],
        ["How does the SEC detect fraud?"]
    ]
)

demo.launch(share=True)  # share=True gives public URL for portfolio